# Step 2 — Generate Product Descriptions

This notebook:
1. Loads the 50-product catalogue (`data/products.csv`)
2. Calls the Nebius AI Studio (OpenAI-compatible) API to generate a 50–90 word description per product
3. Records `generated_description`, `latency_ms`, `input_tokens`, `output_tokens`
4. Adds blank rubric columns for human rating (Step 3)
5. Saves the result to `results/step2_generated_descriptions.xlsx`

**Model:** `meta-llama/Meta-Llama-3.1-8B-Instruct-fast`
**API key:** read from the `NEBIUS_API_KEY` environment variable (see `.env.example`), never hardcoded.

## 0. Install dependencies
Run once if not already installed.

In [2]:
!pip install openai pandas openpyxl python-dotenv

## 1. Load API key

The key is loaded from a local `.env` file, which git ignores. Copy `.env.example` to `.env` and add your key.

In [4]:
from dotenv import load_dotenv
load_dotenv()
print("API key loaded.")

API key loaded.


## 2. Imports & configuration

In [6]:
import os
import time
import pandas as pd
from openai import OpenAI

# ── API configuration ──────────────────────────────────────────────────────────
# Key is loaded from .env file in Cell 1 above
client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)

MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct-fast"

print(f"Client ready. Model: {MODEL}")

Client ready. Model: meta-llama/Meta-Llama-3.1-8B-Instruct-fast


## 3. System prompt

In [8]:
SYSTEM_PROMPT = """You are a professional e-commerce copywriter. Your task is to write a persuasive product description for an online store.

RULES — follow every rule strictly:
1. Length: write between 50 and 90 words. Count carefully. Never go below 50 or above 90.
2. Grounding: use ONLY the information given to you (name, attributes, material, warranty). Do NOT invent features, specifications, or claims that are not in the input.
3. Tone: friendly, confident, and persuasive. Write like a trusted retailer — highlight real benefits, avoid hollow superlatives like "the best ever" or excessive exclamation marks.
4. Fluency: write in natural, flowing prose. No bullet points, no headers, no lists.
5. Grammar: use correct spelling, punctuation, and capitalization throughout.
6. Output: return the product description ONLY. No preamble, no explanation, no word count label — just the description text.

BAD example (do not do this):
"Introducing the MOST AMAZING product you'll ever own!!! It does everything!! Buy now!!!"

GOOD example:
"Built for everyday reliability, the [Product] combines [material] construction with [key feature] to deliver lasting value. Whether you need [use case] or [use case], this product has you covered. Backed by a [warranty] warranty, it's a purchase you can make with confidence."
"""

print("System prompt defined.")

System prompt defined.


## 4. Load the dataset

In [10]:
df_products = pd.read_csv("../data/products.csv")

print(f"Loaded {len(df_products)} products.")
print(f"Columns: {list(df_products.columns)}")
df_products.head(3)

Loaded 50 products.
Columns: ['product_name', 'Product_attribute_list', 'material', 'warranty']


,product_name,Product_attribute_list,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty


## 5. Generation function

In [12]:
def build_user_message(row: pd.Series) -> str:
    """Format product fields into a structured user message."""
    return (
        f"Product name: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}"
    )


def generate_description(row: pd.Series) -> dict:
    """
    Call the Nebius API for a single product row.
    Returns dict with generated_description, latency_ms, input_tokens, output_tokens.
    """
    user_message = build_user_message(row)

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": user_message
                    }
                ]
            }
        ],
        temperature=0.7,
        max_tokens=200,
    )

    latency_ms = (time.perf_counter() - start) * 1000
    description = response.choices[0].message.content.strip()
    usage = response.usage

    return {
        "generated_description": description,
        "latency_ms":            round(latency_ms, 1),
        "input_tokens":          usage.prompt_tokens,
        "output_tokens":         usage.completion_tokens,
    }

print("Generation function defined.")

Generation function defined.


## 6. Test on a single product first

Smoke-test the prompt and the API call on one product before running the full batch of 50.

In [31]:
test_result = generate_description(df_products.iloc[0])
print("=== TEST RESULT ===")
print(f"Description:\n{test_result['generated_description']}")
print(f"\nLatency:       {test_result['latency_ms']} ms")
print(f"Input tokens:  {test_result['input_tokens']}")
print(f"Output tokens: {test_result['output_tokens']}")

=== TEST RESULT ===
Description:
The Apple iPhone 15 Pro is a powerhouse in a compact package. Equipped with the A17 Pro chip and a stunning 120Hz ProMotion display, this phone delivers seamless performance and breathtaking visuals. The titanium frame and Ceramic Shield glass ensure durability and protection. With USB-C fast charging, you can quickly get back to your day. And with a 1-year limited warranty, you can buy with confidence.

Latency:       1032.1 ms
Input tokens:  359
Output tokens: 84


## 7. Run generation across all 50 products

Only run after Cell 6 succeeds.

In [33]:
results = []

for idx, row in df_products.iterrows():
    print(f"[{idx+1:02d}/{len(df_products)}] {row['product_name']} ... ", end="")
    try:
        result = generate_description(row)
        results.append(result)
        print(f"OK  ({result['latency_ms']:.0f} ms, {result['input_tokens']}→{result['output_tokens']} tokens)")
    except Exception as e:
        print(f"ERROR: {e}")
        results.append({
            "generated_description": None,
            "latency_ms":            None,
            "input_tokens":          None,
            "output_tokens":         None,
        })

print(f"\nDone. {sum(1 for r in results if r['generated_description'])} / {len(results)} succeeded.")

[01/50] Apple iPhone 15 Pro ... OK  (865 ms, 359→89 tokens)
[02/50] Samsung Galaxy S24 Ultra ... OK  (1008 ms, 362→85 tokens)
[03/50] Google Pixel 8 Pro ... OK  (943 ms, 360→98 tokens)
[04/50] Sony WH‑1000XM5 Headphones ... OK  (1006 ms, 359→88 tokens)
[05/50] Bose QuietComfort Ultra Earbuds ... OK  (1520 ms, 353→74 tokens)
[06/50] Amazon Echo Dot (5th Gen) ... OK  (941 ms, 349→93 tokens)
[07/50] Dell XPS 13 9310 Laptop ... OK  (898 ms, 367→88 tokens)
[08/50] Apple MacBook Air 13″ (M3) ... OK  (982 ms, 358→94 tokens)
[09/50] Microsoft Surface Pro 10 ... OK  (882 ms, 351→90 tokens)
[10/50] Garmin Forerunner 255 ... OK  (1839 ms, 361→86 tokens)
[11/50] Fitbit Charge 6 ... OK  (1026 ms, 353→97 tokens)
[12/50] GoPro HERO12 Black ... OK  (821 ms, 359→84 tokens)
[13/50] DJI Mini 4 Pro Drone ... OK  (1152 ms, 361→87 tokens)
[14/50] Nintendo Switch OLED ... OK  (784 ms, 352→76 tokens)
[15/50] PlayStation 5 Slim ... OK  (1028 ms, 351→85 tokens)
[16/50] Xbox Series X ... OK  (744 ms, 360→72 toke

## 8. Assemble the output DataFrame

In [35]:
df_results = pd.DataFrame(results)
df_out = pd.concat([df_products.reset_index(drop=True), df_results], axis=1)

# Blank rubric columns — to be filled manually in Step 3
for col in ["fluency", "grammar", "tone", "length", "grounding",
            "latency_rating", "cost_rating", "cost_usd", "final_score"]:
    df_out[col] = None

print(f"DataFrame shape: {df_out.shape}")
print(f"Columns: {list(df_out.columns)}")
df_out.head(3)

DataFrame shape: (50, 17)
Columns: ['product_name', 'Product_attribute_list', 'material', 'warranty', 'generated_description', 'latency_ms', 'input_tokens', 'output_tokens', 'fluency', 'grammar', 'tone', 'length', 'grounding', 'latency_rating', 'cost_rating', 'cost_usd', 'final_score']


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,cost_usd,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Built for everyday reliability, the Apple iPho...",864.7,359,89,None,None,None,None,None,None,None,None,None
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture life's moments with the Samsung Galaxy...,1008.0,362,85,None,None,None,None,None,None,None,None,None
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,"Built for everyday reliability, the Google Pix...",942.9,360,98,None,None,None,None,None,None,None,None,None


## 9. Word count sanity check

In [37]:
df_out["word_count"] = df_out["generated_description"].dropna().apply(lambda t: len(t.split()))

print("Word count distribution:")
print(df_out["word_count"].describe().round(1))
print()
print(f"  Good (50–90):     {((df_out.word_count >= 50) & (df_out.word_count <= 90)).sum()}")
print(f"  Ok   (40–110):    {((df_out.word_count >= 40) & (df_out.word_count <= 110)).sum()}")
print(f"  Bad  (<40|>110):  {((df_out.word_count < 40)  | (df_out.word_count > 110)).sum()}")

Word count distribution:
count    50.0
mean     64.2
std       6.4
min      50.0
25%      60.2
50%      64.0
75%      67.8
max      80.0
Name: word_count, dtype: float64

  Good (50–90):     50
  Ok   (40–110):    50
  Bad  (<40|>110):  0


## 10. Latency & cost summary

In [39]:
print("=== Latency (ms) ===")
print(df_out["latency_ms"].describe().round(1))
avg_latency = df_out["latency_ms"].mean()
latency_rating = "good" if avg_latency <= 3000 else "ok" if avg_latency <= 6000 else "bad"
print(f"\nAverage: {avg_latency:.0f} ms  →  {latency_rating.upper()}")

print("\n=== Tokens & Cost ===")
print(f"Avg input tokens:  {df_out['input_tokens'].mean():.1f}")
print(f"Avg output tokens: {df_out['output_tokens'].mean():.1f}")

# Pricing: $0.03/1M input, $0.09/1M output (Meta-Llama-3.1-8B-Instruct-fast on Nebius)
INPUT_PRICE  = 0.03 / 1_000_000
OUTPUT_PRICE = 0.09 / 1_000_000
avg_cost = df_out["input_tokens"].mean() * INPUT_PRICE + df_out["output_tokens"].mean() * OUTPUT_PRICE
cost_rating = "good" if avg_cost < 0.0005 else "ok" if avg_cost < 0.002 else "bad"
print(f"Avg cost per call: ${avg_cost:.6f} USD  →  {cost_rating.upper()}")

=== Latency (ms) ===
count      50.0
mean     1120.8
std       408.3
min       744.1
25%       884.6
50%       992.8
75%      1149.2
max      2647.7
Name: latency_ms, dtype: float64

Average: 1121 ms  →  GOOD

=== Tokens & Cost ===
Avg input tokens:  355.9
Avg output tokens: 86.4
Avg cost per call: $0.000018 USD  →  GOOD


## 11. Save to Excel

In [50]:
OUTPUT_PATH = "../results/step2_generated_descriptions.xlsx"

df_save = df_out.drop(columns=["word_count"], errors="ignore")
df_save.to_excel(OUTPUT_PATH, index=False, engine="openpyxl")
print(f"Saved {len(df_save)} rows to '{OUTPUT_PATH}'.")
print(f"Columns: {list(df_save.columns)}")

Saved 50 rows to '../results/step2_generated_descriptions.xlsx'.
Columns: ['product_name', 'Product_attribute_list', 'material', 'warranty', 'generated_description', 'latency_ms', 'input_tokens', 'output_tokens', 'fluency', 'grammar', 'tone', 'length', 'grounding', 'latency_rating', 'cost_rating', 'cost_usd', 'final_score']


## 12. Preview — first 5 generated descriptions

In [47]:
for i, row in df_out.head(5).iterrows():
    wc = len(str(row['generated_description']).split())
    print(f"--- Product {i+1}: {row['product_name']} ---")
    print(row["generated_description"])
    print(f"(words: {wc}, latency: {row['latency_ms']} ms)")
    print()

--- Product 1: Apple iPhone 15 Pro ---
Built for everyday reliability, the Apple iPhone 15 Pro combines a durable titanium frame with Ceramic Shield glass to deliver lasting value. With its compact design, it's perfect for on-the-go use. The A17 Pro chip and 120 Hz ProMotion display ensure seamless performance and stunning visuals. Fast charging via USB-C keeps you powered up. Backed by a 1-year limited warranty, it's a purchase you can make with confidence.
(words: 68, latency: 864.7 ms)

--- Product 2: Samsung Galaxy S24 Ultra ---
Capture life's moments with the Samsung Galaxy S24 Ultra. Its 200 MP camera delivers stunning detail, while S-Pen support unlocks new creative possibilities. Enjoy seamless visuals on the 120 Hz AMOLED display, protected by an Armor Aluminum frame and Gorilla Glass Victus. Backed by a 1-year limited warranty, you can rely on this sustainably sourced device to keep up with your active lifestyle.
(words: 63, latency: 1008.0 ms)

--- Product 3: Google Pixel 8 